# Notebook 43 — Robust Engineering Preserves the Largest Connected Admissible Region

**Engineering statement**

> A physical platform does not compute because one operating point succeeds.  
> It computes because a connected admissible region exists around that operating point.

Notebook 37 introduced operating regimes as intersections of active specifications.  
This upgraded Notebook 43 turns that language into a quantitative robustness notebook.

The central object is:

\[
\Omega = \{x : \text{all active specifications admit } x\}
\]

and the design target is not just a point \(x_0\), but the largest connected component:

\[
\Omega_{\max}.
\]

Engineering then maximizes:

1. admissible area,
2. connectedness,
3. distance to every failure boundary,
4. sensitivity improvement per engineering action.

## Setup

This notebook is self-contained. It generates figures, tables, JSON summaries, and a zip bundle.

Outputs:

```text
figures/
results/csv/
results/json/
results/43_outputs_upgraded.zip
```

In [ ]:
from pathlib import Path
import json
import zipfile
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from matplotlib.colors import ListedColormap

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

def save_table(df, stem):
    csv_path = CSV / f"{stem}.csv"
    json_path = JS / f"{stem}.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)
    print(f"saved: {csv_path}")
    print(f"saved: {json_path}")

print("ROOT:", ROOT)

## 1. Region-first design

The operating point should be selected **inside** the largest connected admissible region.

The region comes first:

```text
largest connected admissible region
        ↓
choose operating point
        ↓
maintain margin
        ↓
compute reliably
```

That reverses the usual point-first optimization story.

In [ ]:
x_main, y_main, X_main, Y_main = grid(280)
Z_main = admissibility_score(X_main, Y_main)
threshold = 0.50

fig, ax = plt.subplots(figsize=(10.5, 6.2))
im = ax.imshow(Z_main, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
cs = ax.contour(X_main, Y_main, Z_main, levels=[0.35, 0.55, 0.72], colors="black", linewidths=[1.4, 2.0, 2.8])
ax.clabel(cs, inline=True, fontsize=9)

chosen_point = (0.60, 0.22)
ax.scatter([chosen_point[0]], [chosen_point[1]], s=260, zorder=4, color="orange")
ax.add_patch(Circle(chosen_point, 0.12, fill=False, linestyle="--", linewidth=2.3, color="black"))
ax.annotate("chosen operating point", xy=chosen_point, xytext=(0.72, 0.34),
            arrowprops=dict(arrowstyle="->", linewidth=2.2), fontsize=11)

ax.text(0.50, 0.50, "largest connected\nadmissible region", ha="center", fontsize=15)
ax.text(0.68, 0.18, "maximum\nrobustness margin", fontsize=11)
ax.text(0.16, 0.82, "under-driven", ha="center", fontsize=11)
ax.text(0.84, 0.82, "unstable /\nover-driven", ha="center", fontsize=11)
ax.text(0.88, 0.08, "loss-limited", ha="center", fontsize=11)

ax.set_title("Region-First Design: Choose the Point Inside the Region", fontsize=17)
ax.set_xlabel("squeezing / nonlinear interaction strength")
ax.set_ylabel("optical loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("admissibility score")

op = chosen_point
savefig(fig, "43_01_region_first_design.png")
plt.show()

### Engineering observation

The operating point is only one realization. The specification is the region that admits many nearby realizations. Robustness is the ability to remain inside that region under perturbation.

## 2. Connected-component metrics

This cell computes topology metrics from simulated admissible regions.

For each architecture we derive:

- admitted area,
- largest connected component,
- component count,
- average margin proxy,
- critical failure mode.

The architecture comparison is now model-derived rather than assigned by hand.

In [ ]:
def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    h, w = mask.shape
    comps = []
    for i in range(h):
        for j in range(w):
            if mask[i,j] and not visited[i,j]:
                q = deque([(i,j)])
                visited[i,j] = True
                pts = []
                while q:
                    a,b = q.popleft()
                    pts.append((a,b))
                    for da, db in [(1,0),(-1,0),(0,1),(0,-1)]:
                        na, nb = a+da, b+db
                        if 0 <= na < h and 0 <= nb < w and mask[na,nb] and not visited[na,nb]:
                            visited[na,nb] = True
                            q.append((na,nb))
                comps.append(pts)
    comps.sort(key=len, reverse=True)
    return comps

def architecture_region(name, n=240):
    x, y, X, Y = grid(n)
    if name == "single resonator":
        Z = admissibility_score(X, Y, drive_shift=-0.04, loss_scale=1.25, timing_scale=1.20)
        Z *= np.exp(-2.8*np.maximum(X-0.75,0)**2)
        critical = "calibration"
    elif name == "coupled resonators":
        Z = admissibility_score(X, Y, drive_shift=-0.02, loss_scale=1.10, timing_scale=1.05)
        Z *= (0.94 + 0.06*np.cos(12*X))
        critical = "loss"
    elif name == "programmable mesh":
        Z = admissibility_score(X, Y, drive_shift=0.02, loss_scale=1.0, timing_scale=0.95)
        Z *= (0.96 + 0.04*np.cos(8*Y))
        critical = "timing"
    elif name == "hybrid chip":
        Z = admissibility_score(X, Y, drive_shift=0.03, loss_scale=0.80, timing_scale=0.75)
        Z = np.clip(Z * 1.12, 0, 1)
        critical = "none"
    else:
        raise ValueError(name)
    Z = Z / Z.max()
    return x, y, X, Y, Z, critical

def boundary_distance(mask):
    # simple Manhattan distance transform from false cells, no scipy required
    h, w = mask.shape
    INF = h + w + 10
    dist = np.full((h,w), INF, dtype=float)
    q = deque()
    for i in range(h):
        for j in range(w):
            if not mask[i,j]:
                dist[i,j] = 0
                q.append((i,j))
    while q:
        i,j = q.popleft()
        for di,dj in [(1,0),(-1,0),(0,1),(0,-1)]:
            ni,nj = i+di,j+dj
            if 0 <= ni < h and 0 <= nj < w and dist[ni,nj] > dist[i,j] + 1:
                dist[ni,nj] = dist[i,j] + 1
                q.append((ni,nj))
    return dist

architectures = ["single resonator", "coupled resonators", "programmable mesh", "hybrid chip"]
metrics = []

for arch in architectures:
    x, y, X, Y, Za, critical = architecture_region(arch)
    mask = Za >= threshold
    comps = connected_components(mask)
    area = mask.mean()
    largest = len(comps[0]) / mask.size if comps else 0
    component_count = len(comps)
    dist = boundary_distance(mask)
    if comps:
        largest_pts = np.array(comps[0])
        avg_margin = dist[largest_pts[:,0], largest_pts[:,1]].mean() / max(mask.shape)
        max_margin = dist[largest_pts[:,0], largest_pts[:,1]].max() / max(mask.shape)
    else:
        avg_margin = 0
        max_margin = 0
    metrics.append({
        "architecture": arch,
        "admitted_area": round(area, 3),
        "largest_component": round(largest, 3),
        "component_count": component_count,
        "average_margin": round(avg_margin, 3),
        "maximum_margin": round(max_margin, 3),
        "critical_failure": critical
    })

topology_metrics = pd.DataFrame(metrics)
save_table(topology_metrics, "43_topology_metrics")
topology_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.axis("off")

columns = ["Architecture", "Largest component", "Admissible area", "Average margin", "Components", "Critical failure"]
rows = []
for _, r in topology_metrics.iterrows():
    rows.append([
        r["architecture"],
        f'{r["largest_component"]:.3f}',
        f'{r["admitted_area"]:.3f}',
        f'{r["average_margin"]:.3f}',
        str(int(r["component_count"])),
        r["critical_failure"],
    ])

table = ax.table(cellText=rows, colLabels=columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1.05, 1.7)
for (r,c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")
    if c == 0 and r > 0:
        cell.set_text_props(ha="left")

ax.set_title("Topology Metrics of the Admissible Region", fontsize=17, pad=18)
ax.text(0.5, -0.08, "Architecture ranking is derived from simulated admissible-region topology.",
        ha="center", transform=ax.transAxes, fontsize=11)

savefig(fig, "43_02_topology_metrics_table.png")
plt.show()

## 3. Region connectivity under perturbation

A region can shrink or disconnect under perturbation. Disconnection matters because computation needs connected paths, routing, measurement order, and feed-forward.

In [ ]:
def gaussian_blob(X, Y, cx, cy, sx, sy, amp=1.0):
    return amp * np.exp(-((X-cx)**2/(2*sx**2) + (Y-cy)**2/(2*sy**2)))

def region_case(kind, n=230):
    x = np.linspace(0,1,n)
    y = np.linspace(0,1,n)
    X,Y = np.meshgrid(x,y)
    if kind == "baseline":
        Z = (gaussian_blob(X,Y,0.48,0.28,0.18,0.16,1.0)
             + gaussian_blob(X,Y,0.62,0.30,0.18,0.16,0.9))
    elif kind == "shrunk":
        Z = gaussian_blob(X,Y,0.55,0.28,0.15,0.12,1.0)
    elif kind == "disconnected":
        Z = (gaussian_blob(X,Y,0.28,0.28,0.08,0.08,1.0)
             + gaussian_blob(X,Y,0.52,0.36,0.08,0.08,0.95)
             + gaussian_blob(X,Y,0.80,0.24,0.08,0.08,1.0))
    else:
        raise ValueError(kind)
    Z = Z / Z.max()
    return x,y,X,Y,Z

cases = [("baseline","Baseline"), ("shrunk","Shrunk"), ("disconnected","Disconnected")]
region_rows = []
fig, axes = plt.subplots(1,3,figsize=(14,4.4),sharex=True,sharey=True)
for ax, (kind, title) in zip(axes, cases):
    x,y,Xc,Yc,Zc = region_case(kind)
    mask = Zc >= 0.5
    comps = connected_components(mask)
    largest = len(comps[0])/mask.size if comps else 0
    area = mask.mean()
    region_rows.append({"case":kind, "area":area, "largest_component":largest, "component_count":len(comps)})
    ax.imshow(Zc, origin="lower", extent=[0,1,0,1], vmin=0, vmax=1, aspect="auto")
    ax.contour(Xc,Yc,Zc,levels=[0.5],colors="black",linewidths=2.4)
    ax.set_title(title)
    ax.set_xlabel("drive")
    ax.text(0.05,0.90,f"components: {len(comps)}",transform=ax.transAxes,fontsize=10)
    ax.text(0.05,0.82,f"largest: {largest:.2f}",transform=ax.transAxes,fontsize=10)
    if ax is axes[0]:
        ax.set_ylabel("loss")
fig.suptitle("Region Connectivity Under Perturbation", fontsize=17, y=1.03)
savefig(fig, "43_03_region_connectivity.png")
plt.show()

region_connectivity = pd.DataFrame(region_rows)
save_table(region_connectivity, "43_region_connectivity")
region_connectivity

## 4. Sensitivity of the largest connected region

Engineering priority should be derived from sensitivity:

\[
S_i = rac{\partial A_{\max}}{\partial p_i},
\]

where \(A_{\max}\) is the size of the largest connected admissible component.

This asks:

> Which engineering action buys the most robust region?

In [ ]:
def largest_component_area_from_field(Z_field, threshold=0.50):
    Z_norm = Z_field / np.nanmax(Z_field)
    mask = Z_norm >= threshold
    comps = connected_components(mask)
    return len(comps[0]) / mask.size if comps else 0.0

def perturbation_field(action, n=220):
    x_s, y_s, X_s, Y_s = grid(n)
    base_Z = admissibility_score(X_s, Y_s)

    if action == "loss reduction":
        # Expands vertically into higher-loss territory.
        improved = admissibility_score(X_s, np.clip(Y_s * 0.86, 0, 1))
    elif action == "timing control":
        # Reduces timing shear near high-drive / high-loss boundary.
        improved = admissibility_score(X_s, Y_s, timing_scale=0.70)
    elif action == "calibration stabilization":
        # Recenters a mildly shifted operating landscape.
        baseline = admissibility_score(np.clip(X_s + 0.035, 0, 1), Y_s, calibration_shift=0.08)
        improved = admissibility_score(X_s, Y_s, calibration_shift=0.02)
        return baseline, improved
    elif action == "detection improvement":
        # Raises readout-admitted area without changing resource generation.
        readout_gate = 0.90 + 0.10 * np.exp(-((X_s-0.60)**2/0.12 + (Y_s-0.18)**2/0.08))
        improved = np.clip(base_Z * 1.06 * readout_gate, 0, 1)
    elif action == "routing improvement":
        # Smooths small connectivity defects.
        defect = 1 - 0.08*np.sin(10*X_s)*np.sin(8*Y_s)
        baseline = np.clip(base_Z * defect, 0, 1)
        improved = np.clip(base_Z * (1 - 0.02*np.sin(10*X_s)*np.sin(8*Y_s)), 0, 1)
        return baseline, improved
    elif action == "local optimization":
        # Improves a point neighborhood but only modestly expands the region.
        bump = 1 + 0.10*np.exp(-((X_s-0.60)**2/0.018 + (Y_s-0.22)**2/0.018))
        improved = np.clip(base_Z * bump, 0, 1)
    else:
        raise ValueError(action)

    return base_Z, improved

actions = [
    "loss reduction",
    "timing control",
    "calibration stabilization",
    "detection improvement",
    "routing improvement",
    "local optimization",
]

sens = []
for action in actions:
    z0, z1 = perturbation_field(action)
    a0 = largest_component_area_from_field(z0)
    a1 = largest_component_area_from_field(z1)
    sens.append({
        "engineering_action": action,
        "delta_largest_component": max(0, a1-a0),
        "baseline": a0,
        "improved": a1,
    })

sensitivity = pd.DataFrame(sens).sort_values("delta_largest_component", ascending=False)
save_table(sensitivity, "43_sensitivity_largest_component")

fig, ax = plt.subplots(figsize=(9.5,5.6))
splot = sensitivity.iloc[::-1]
ax.barh(splot["engineering_action"], splot["delta_largest_component"])
ax.set_xlabel("increase in largest connected admissible region")
ax.set_title("Sensitivity of the Largest Connected Admissible Region", fontsize=17)
ax.grid(axis="x", alpha=0.25)
xmax = max(splot["delta_largest_component"].max(), 0.001)
ax.set_xlim(0, xmax*1.18)
for yv, val in zip(splot["engineering_action"], splot["delta_largest_component"]):
    ax.text(val + xmax*0.015, yv, f"{val:.3f}", va="center", fontsize=10)
savefig(fig, "43_04_sensitivity_largest_component.png")
plt.show()

sensitivity

### Engineering observation

This figure replaces subjective priority scores. The priority ranking comes from how much each engineering action expands the largest connected admissible component.

## 5. Robustness phase diagram

A design can be:

- connected and robust,
- connected but fragile,
- disconnected,
- disconnected and unstable.

The useful target is the connected robust phase.

In [ ]:
# Build a phase diagram from region size and margin proxies.
R = np.linspace(0,1,240)  # connected component fraction
M = np.linspace(0,1,240)  # margin
RR, MM = np.meshgrid(R, M)

phase_score = 0.65*RR + 0.35*MM
connected = RR > 0.35
robust = MM > 0.35
phase = np.zeros_like(RR)
phase[(connected) & (~robust)] = 1
phase[(~connected) & (robust)] = 2
phase[(connected) & (robust)] = 3

fig, ax = plt.subplots(figsize=(8.5,6.4))
cmap = ListedColormap(["#d9d9d9", "#9ecae1", "#c7e9c0", "#41ab5d"])
im = ax.imshow(phase, origin="lower", extent=[0,1,0,1], aspect="auto", cmap=cmap, vmin=0, vmax=3)

ax.axvline(0.35, color="black", linewidth=2)
ax.axhline(0.35, color="black", linewidth=2)
ax.text(0.08,0.12,"disconnected\\nfragile",fontsize=12)
ax.text(0.62,0.12,"connected\\nfragile",fontsize=12)
ax.text(0.08,0.70,"disconnected\\nbuffered",fontsize=12)
ax.text(0.61,0.70,"connected\\nrobust",fontsize=13, weight="bold")

# trajectories
ax.plot([0.18,0.42,0.62,0.78],[0.25,0.38,0.52,0.66], marker="o", linewidth=2.5)
ax.annotate("design A", xy=(0.78,0.66), xytext=(0.65,0.82), arrowprops=dict(arrowstyle="->", linewidth=1.8))

ax.plot([0.55,0.50,0.43,0.31],[0.55,0.45,0.31,0.20], marker="o", linewidth=2.5, linestyle="--")
ax.annotate("design B crosses boundary", xy=(0.33,0.20), xytext=(0.45,0.08), arrowprops=dict(arrowstyle="->", linewidth=1.8))

ax.set_xlabel("largest connected component")
ax.set_ylabel("minimum robustness margin")
ax.set_title("Robustness Phase Diagram", fontsize=17)

savefig(fig, "43_05_robustness_phase_diagram.png")
plt.show()

## 6. Failure boundary geometry

Robustness is distance to every active failure boundary.

\[
d(x) = \min_{b \in \partial \Omega} \|x-b\|.
\]

The shortest distance controls the margin.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,8.5))
ax.set_aspect("equal")
ax.axis("off")

center = np.array([0.5,0.5])
boundaries = {
    "loss": np.array([-0.42, 0.00]),
    "timing": np.array([0.42, 0.00]),
    "detector": np.array([0.00, -0.34]),
    "calibration": np.array([0.00, 0.44]),
    "phase": np.array([0.28, 0.30]),
    "routing": np.array([-0.25, -0.26]),
}
active_label, active_vec = min(boundaries.items(), key=lambda kv: np.linalg.norm(kv[1]))
active_radius = np.linalg.norm(active_vec)

ax.scatter([center[0]],[center[1]],s=250,zorder=5)
ax.text(center[0], center[1]-0.06, "operating\npoint", ha="center", va="top", fontsize=10)

for label, v in boundaries.items():
    end = center + v
    is_active = label == active_label
    color = "red" if is_active else "black"
    lw = 3.0 if is_active else 2.0
    ax.annotate("", xy=end, xytext=center, arrowprops=dict(arrowstyle="->", linewidth=lw, color=color))
    ax.scatter([end[0]],[end[1]],s=90 if is_active else 70, color=color if is_active else None, zorder=4)
    ax.text(end[0] + 0.03*np.sign(v[0] if v[0] != 0 else 1),
            end[1] + 0.03*np.sign(v[1] if v[1] != 0 else 1),
            label, fontsize=11, color=color)
    ax.text(*(center + 0.48*v), f"{np.linalg.norm(v):.2f}", fontsize=9, color=color)

ax.add_patch(Circle(center, active_radius, fill=False, linestyle="--", linewidth=2.6, color="red"))
ax.text(0.5, 0.93, f"active margin set by: {active_label}", ha="center", fontsize=12, color="red")

ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_title("Failure Boundary Geometry: Robustness Is the Minimum Distance", fontsize=16)

savefig(fig, "43_06_failure_boundary_geometry.png")
plt.show()

## 7. Distance-to-boundary field

Now compute the distance field inside the admitted region.

Bright regions are robust interior points. Dark regions are near failure boundaries.

In [ ]:
mask_distance = Z_main >= threshold
dist = boundary_distance(mask_distance)
dist_norm = dist / dist.max()
dist_masked = np.where(mask_distance, dist_norm, np.nan)

# Find max distance point inside admitted region.
idx = np.nanargmax(dist_masked)
iy, ix = np.unravel_index(idx, dist_masked.shape)
best_x = x_main[ix]
best_y = y_main[iy]

fig, ax = plt.subplots(figsize=(10,6.2))
im = ax.imshow(dist_masked, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
ax.contour(X_main, Y_main, Z_main, levels=[threshold], colors="black", linewidths=2.2)

# orange = chosen operating point; red = maximum-margin point
ax.scatter([op[0]],[op[1]],s=160,zorder=4, color="orange")
ax.text(op[0]+0.02, op[1]-0.04, "chosen operating point", fontsize=10)

ax.scatter([best_x],[best_y],s=240,zorder=4, color="red")
ax.annotate("maximum-margin point", xy=(best_x,best_y), xytext=(best_x+0.12,best_y+0.20),
            arrowprops=dict(arrowstyle="->", linewidth=2.0), fontsize=11)

ax.annotate("", xy=(best_x,best_y), xytext=op,
            arrowprops=dict(arrowstyle="->", linewidth=2.0, linestyle="--"))

ax.set_title("Distance to Failure Boundary Inside the Admitted Region", fontsize=17)
ax.set_xlabel("squeezing / nonlinear interaction strength")
ax.set_ylabel("optical loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("normalized distance to boundary")
savefig(fig, "43_07_distance_to_boundary.png")
plt.show()

distance_metrics = pd.DataFrame([{
    "maximum_margin_x": best_x,
    "maximum_margin_y": best_y,
    "chosen_point_x": op[0],
    "chosen_point_y": op[1],
    "max_normalized_distance": np.nanmax(dist_masked),
}])
save_table(distance_metrics, "43_distance_to_boundary")
distance_metrics

### Engineering observation

The maximum-margin point is not necessarily the highest-admissibility point. This is the key distinction between performance optimization and robust engineering.

## 8. Design landscape: prefer plateaus over ridges

A narrow ridge can score high while being fragile.

A plateau may score lower at its peak but preserve computation under variation.

In [ ]:
xx = np.linspace(-1.2,1.2,320)
yy = np.linspace(-1.2,1.2,320)
XX,YY = np.meshgrid(xx,yy)

plateau = 0.72*np.exp(-((XX+0.20)**8)/0.18)*np.exp(-((YY+0.05)**8)/0.08)
ridge = 1.1*np.exp(-((XX-0.53)**2/0.018 + (YY+0.02)**2/0.34))
cliff = 1/(1+np.exp(9*(YY-0.70)))
L = (plateau + ridge)*cliff
L = L/L.max()

fig, ax = plt.subplots(figsize=(9.5,6.2))
im = ax.imshow(L, origin="lower", extent=[xx.min(),xx.max(),yy.min(),yy.max()], aspect="auto", vmin=0, vmax=1)
cs = ax.contour(XX,YY,L,levels=[0.35,0.55,0.75],colors="black",linewidths=[1.5,2.0,2.6])
ax.clabel(cs, inline=True, fontsize=9)
ax.text(-0.55,0.05,"safe plateau",fontsize=14)
ax.text(0.54,-0.16,"narrow ridge",fontsize=14)
ax.text(0.32,0.84,"unstable cliff",fontsize=13)
ax.annotate("good engineering\\nwidens this plateau", xy=(-0.36,0.17), xytext=(-1.05,0.70),
            arrowprops=dict(arrowstyle="->", linewidth=2.2), fontsize=12)
ax.set_title("Design Landscape: Prefer a Plateau Over a Ridge", fontsize=17)
ax.set_xlabel("control parameter 1")
ax.set_ylabel("control parameter 2")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("relative computation support")
savefig(fig, "43_08_design_landscape.png")
plt.show()

## 9. Specification ladder

The summary is now a specification ladder.

Each stage preserves an object needed by the next stage.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5,4.8))
ax.axis("off")

ladder = [
    ("Physics", "optical resources"),
    ("Admissibility", "admitted resources"),
    ("Largest connected\nadmissible region", "topological engineering object"),
    ("Operating point", "chosen realization with margin"),
    ("Fault-tolerant\ncomputation", "logical execution"),
]
xs = np.linspace(0.10,0.90,len(ladder))
y = 0.55
w,h = 0.16,0.30

for i,(stage,obj) in enumerate(ladder):
    ax.add_patch(Rectangle((xs[i]-w/2,y-h/2),w,h,fill=False,linewidth=2.2))
    ax.text(xs[i], y+0.045, stage, ha="center", va="center", fontsize=10.5, weight="bold")
    ax.text(xs[i], y-0.075, obj, ha="center", va="center", fontsize=8.8)
    if i < len(ladder)-1:
        ax.annotate("", xy=(xs[i+1]-w/2-0.012,y), xytext=(xs[i]+w/2+0.012,y),
                    arrowprops=dict(arrowstyle="->", linewidth=2.0))
ax.set_title("Specification Ladder: The Region Specifies the Operating Point", fontsize=17)
ax.text(0.5,0.16,"The specification is the largest connected admissible region; the operating point is one admitted realization inside it.",
        ha="center", fontsize=11)
savefig(fig, "43_09_specification_ladder.png")
plt.show()

## 10. Failure tree

Logical failure follows from an earlier engineering failure: the connected admissible region disconnects.

In [ ]:
fig, ax = plt.subplots(figsize=(11,6))
ax.axis("off")

leaves = [
    ("loss",(0.10,0.82)),
    ("phase drift",(0.10,0.64)),
    ("detector noise",(0.10,0.46)),
    ("timing jitter",(0.10,0.28)),
    ("feed-forward delay",(0.10,0.10)),
]
mid = (0.56,0.46)
out = (0.88,0.46)

for label,pos in leaves:
    ax.add_patch(Rectangle((pos[0]-0.085,pos[1]-0.045),0.17,0.09,fill=False,linewidth=1.8))
    ax.text(*pos,label,ha="center",va="center",fontsize=10)
    ax.annotate("", xy=(mid[0]-0.17,mid[1]), xytext=(pos[0]+0.09,pos[1]),
                arrowprops=dict(arrowstyle="->", linewidth=1.4, alpha=0.75))
ax.add_patch(Rectangle((mid[0]-0.17,mid[1]-0.075),0.34,0.15,fill=False,linewidth=2.4))
ax.text(*mid,"connected admissible\\nregion disconnects",ha="center",va="center",fontsize=11)

ax.add_patch(Rectangle((out[0]-0.09,out[1]-0.045),0.18,0.09,fill=False,linewidth=1.9))
ax.text(*out,"logical failure",ha="center",va="center",fontsize=11)
ax.annotate("", xy=(out[0]-0.10,out[1]), xytext=(mid[0]+0.18,mid[1]),
            arrowprops=dict(arrowstyle="->", linewidth=2.2))

ax.set_title("Failure Tree: Logical Failure Follows Disconnection of the Region", fontsize=17)
ax.text(0.5,-0.03,"robustness protects the intermediate engineering object before computation fails",
        ha="center", transform=ax.transAxes, fontsize=11)
savefig(fig, "43_10_failure_tree.png")
plt.show()

## 11. Engineering principle

A physical platform does not compute because one operating point succeeds.

It computes because a connected admissible region exists around that operating point.

Engineering therefore maximizes:

1. admissible area,
2. connectedness,
3. distance to every failure boundary,

before optimizing logical performance.

In [ ]:
principle = pd.DataFrame([
    {"Object": "Admissible area", "Meaning": "How many resources satisfy active specifications?", "Metric": "area(Omega)"},
    {"Object": "Connectedness", "Meaning": "Do admitted resources form usable paths?", "Metric": "largest connected component"},
    {"Object": "Boundary distance", "Meaning": "How far is the operating point from failure?", "Metric": "min distance to boundary"},
    {"Object": "Sensitivity", "Meaning": "Which engineering action expands the region?", "Metric": "partial derivative of component size"},
    {"Object": "Logical performance", "Meaning": "What computation remains after constraints?", "Metric": "fidelity, depth, success probability"},
])
save_table(principle, "43_engineering_principle")

fig, ax = plt.subplots(figsize=(14,4.2))
ax.axis("off")
table = ax.table(cellText=principle.values, colLabels=principle.columns, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1.05,1.65)
for (r,c), cell in table.get_celld().items():
    cell.set_linewidth(1.1)
    if r == 0:
        cell.set_text_props(weight="bold")
ax.set_title("Engineering Principle: Optimize After Preserving the Region", fontsize=17, pad=18)
savefig(fig, "43_11_engineering_principle.png")
plt.show()
principle

## 12. Engineering specification chain

Final synthesis: each stage specifies what the next stage must preserve.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5,4.8))
ax.axis("off")

chain = [
    ("physics", "what can exist?"),
    ("resources", "what was generated?"),
    ("admissibility", "what survives?"),
    ("largest connected\nadmissible region", "what remains connected?"),
    ("robust operating\npoint", "what has margin?"),
    ("logical\ncomputation", "what can execute?"),
]

xs = np.linspace(0.08,0.92,len(chain))
y = 0.56
w,h = 0.13,0.25

for i,(title,subtitle) in enumerate(chain):
    ax.add_patch(Rectangle((xs[i]-w/2,y-h/2),w,h,fill=False,linewidth=2.3))
    ax.text(xs[i], y+0.045, title, ha="center", va="center", fontsize=10.5, weight="bold")
    ax.text(xs[i], y-0.070, subtitle, ha="center", va="center", fontsize=8.8)
    if i < len(chain)-1:
        ax.annotate("", xy=(xs[i+1]-w/2-0.008,y), xytext=(xs[i]+w/2+0.008,y),
                    arrowprops=dict(arrowstyle="->", linewidth=2.2))

ax.text(0.5,0.20,"Engineering improves computation by preserving the region before optimizing the point.",
        ha="center", fontsize=12)
ax.set_title("Engineering Specification Chain", fontsize=18)
savefig(fig, "43_12_engineering_specification_chain.png")
plt.show()

## 13. Export bundle

In [ ]:
bundle = {
    "topology_metrics": topology_metrics.to_dict(orient="records"),
    "region_connectivity": region_connectivity.to_dict(orient="records"),
    "sensitivity": sensitivity.to_dict(orient="records"),
    "distance_metrics": distance_metrics.to_dict(orient="records"),
    "engineering_principle": principle.to_dict(orient="records"),
}
bundle_path = JS / "43_upgraded_review_bundle.json"
with open(bundle_path, "w", encoding="utf-8") as f:
    json.dump(bundle, f, indent=2)
print(f"saved: {bundle_path}")

zip_path = RES / "43_outputs_upgraded.zip"
files_to_zip = list(FIG.glob("43_*.png")) + list(CSV.glob("43_*.csv")) + list(JS.glob("43_*.json"))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))